# 별의 끝에서 — 별빛 세계 배경 생성 (교내 GPU 틸론 · SDXL-Turbo)

해커톤 실습 노트북(workshop.ipynb)의 **0-1 환경 확인**, **3-1 SDXL 준비** 코드를 그대로 사용하고, 마지막 셀에서 《별의 끝에서》 배경 6장을 생성합니다.
위에서부터 **Shift + Enter**. 결과: `results/starlight/` 와 `starlight_outputs.zip`

## 1. 환경 확인 (workshop 0-1)

In [ ]:
import os
import sys
from importlib import metadata

print('Python:', sys.version.split()[0])
for package in ('torch', 'transformers', 'diffusers', 'accelerate', 'huggingface-hub', 'imageio', 'imageio-ffmpeg'):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, '없음')
try:
    import torch
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU 전체 메모리(GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
        free, total = torch.cuda.mem_get_info()
        print('지금 사용 가능한 GPU 메모리(GB):', round(free / 1024**3, 2), '← 다른 사용자와 공유 중이면 전체보다 작습니다')
except ImportError:
    print('torch를 불러올 수 없어 GPU를 확인하지 못했습니다.')
for name in ['qwen2.5-1.5b-instruct', 'sdxl-turbo', 'musicgen-small', 'stable-video-diffusion-img2vid']:
    found = [p for p in (os.path.join('/opt/models', name), os.path.expanduser(os.path.join('~/models', name))) if os.path.isdir(p)]
    print('모델:', name, '확인됨 ' + found[0] if found else '없음: 준비 셀에서 자동으로 내려받습니다')
try:
    import imageio_ffmpeg
    print('ffmpeg(영상 저장용):', imageio_ffmpeg.get_ffmpeg_exe())
except Exception as err:  # 영상 실습을 하지 않으면 없어도 됩니다.
    print('ffmpeg(영상 저장용): 없음 -', type(err).__name__)

## 2. SDXL-Turbo 준비 (workshop 3-1, 서버의 /opt/models 사용)

In [ ]:
import json
import time
import uuid
from pathlib import Path
import torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

def prepare_model(repo_id, folder, marker, patterns):
    """서버 배포본을 먼저 쓰고, 없으면 내 홈 폴더에 한 번만 내려받습니다."""
    from pathlib import Path
    from huggingface_hub import snapshot_download
    shared = Path('/opt/models') / folder
    if (shared / marker).is_file():
        return str(shared)
    target = Path.home() / 'models' / folder
    done = target / '.download_complete'
    if done.is_file():
        return str(target)
    target.mkdir(parents=True, exist_ok=True)
    print('서버에 모델이 없어 내려받습니다. 몇 분 걸릴 수 있습니다:', target)
    try:
        snapshot_download(repo_id, local_dir=str(target), allow_patterns=patterns)
    except Exception as exc:
        raise RuntimeError(f'{repo_id} 모델을 내려받지 못했습니다. 인터넷 연결과 디스크 공간을 확인하고 스태프에게 알려 주세요.') from exc
    done.touch()
    return str(target)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU를 사용할 수 없습니다. 강사에게 문의하세요.')
if any(name in globals() for name in ('qwen_model', 'sdxl_pipe', 'img2img_pipe', 'music_model', 'video_pipe')):
    raise RuntimeError('기존 모델이 있습니다. 먼저 모델 해제 셀을 실행하세요.')

sdxl_path = prepare_model('stabilityai/sdxl-turbo', 'sdxl-turbo', 'model_index.json',
                          ['*.json', '*.txt', '*.fp16.safetensors'])
sdxl_pipe = AutoPipelineForText2Image.from_pretrained(
    sdxl_path, torch_dtype=torch.float16, variant='fp16', local_files_only=True
).to('cuda')
print('SDXL Turbo 준비 완료')
print('GPU 점유 메모리(GB):', round(torch.cuda.memory_allocated() / 1024**3, 2), '← 슬라이드 18번 그림과 비교해 보세요')

## 3. 별빛 세계 배경 생성 + GPU 사용 기록 + 웹용 WebP + zip

In [ ]:
# ✦ 별의 끝에서 — 별빛 세계 배경 생성 (먼저 셀 0-1 환경 확인, 셀 3-1 SDXL 준비를 실행하세요)
import json, time, datetime, subprocess, shutil
from pathlib import Path
import torch
from PIL import Image
from IPython.display import display

if 'sdxl_pipe' not in globals():
    raise RuntimeError('먼저 셀 3-1(SDXL 준비)을 실행하세요.')

STYLE = ('soft watercolor illustration, 2.5D game background, dreamy starlight world, '
         'pastel purple, deep navy and warm gold palette, glowing stars, gentle light, wide landscape, no people, no text')
PROMPTS = {
    'title':  'a vast night sky full of stars over a quiet horizon, one bright star at the end of the sky',
    'house':  'inside a cozy old star house, tall arched windows showing the night sky, wooden bookshelves, hanging star lanterns',
    'garden': 'a forgotten garden at dusk, weeping willow trees, flower arches, glowing flowers, soft mist',
    'plain':  'a wide windy plain at twilight, drifting soft shadows, scattered points of light on the grass',
    'attic':  'an attic of time, old clocks, unopened boxes, an old map and a tall mirror, warm dust in moonlight',
    'hall':   'a grand hall of stars, constellations glowing on a domed ceiling, light gathering at an altar',
}
W, H, STEPS = 896, 512, 4   # SDXL-Turbo: guidance 0, 1~4 steps

out = Path('results/starlight'); (out / 'web').mkdir(parents=True, exist_ok=True)
props = torch.cuda.get_device_properties(0)
log = {'model': 'stabilityai/sdxl-turbo', 'model_path': str(sdxl_path), 'gpu': torch.cuda.get_device_name(0),
       'gpu_total_gb': round(props.total_memory / 1024**3, 2), 'cuda': torch.version.cuda, 'torch': torch.__version__,
       'size': [W, H], 'steps': STEPS, 'guidance_scale': 0.0, 'style': STYLE,
       'started_at': datetime.datetime.now().isoformat(timespec='seconds'), 'images': []}
torch.cuda.reset_peak_memory_stats()
for i, (name, prompt) in enumerate(PROMPTS.items()):
    seed = 2026 + i
    generator = torch.Generator(device='cpu').manual_seed(seed)
    torch.cuda.synchronize(); start = time.perf_counter()
    with torch.inference_mode():
        image = sdxl_pipe(prompt=f'{prompt}, {STYLE}', generator=generator, num_inference_steps=STEPS,
                          guidance_scale=0.0, width=W, height=H, num_images_per_prompt=1).images[0]
    torch.cuda.synchronize(); seconds = round(time.perf_counter() - start, 3)
    image.save(out / f'{name}.png')
    im = image.convert('RGB'); w, h = im.size; nh = int(w * 9 / 16)
    im.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh)).resize((1600, 900), Image.LANCZOS).save(out / 'web' / f'{name}.webp', 'WEBP', quality=85)
    log['images'].append({'name': name, 'prompt': prompt, 'seed': seed, 'seconds': seconds})
    print(f'{name}: {seconds}초'); display(image.resize((448, 256)))
log['peak_gpu_memory_gb'] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
log['finished_at'] = datetime.datetime.now().isoformat(timespec='seconds')
try:
    log['nvidia_smi'] = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=30).stdout
except Exception as e:
    log['nvidia_smi'] = f'실행 불가: {e}'
(out / 'generation_log.json').write_text(json.dumps(log, ensure_ascii=False, indent=2), encoding='utf-8')
shutil.make_archive('starlight_outputs', 'zip', out)
print('완료! 최대 GPU 메모리', log['peak_gpu_memory_gb'], 'GB → 왼쪽 파일 목록의 starlight_outputs.zip 을 내려받아 주세요.')

# ── 결과 요약 화면 (캡처용) ──────────────────────────────
import base64, io
from IPython.display import HTML
from PIL import ImageDraw
LABELS = {'title': '타이틀', 'house': '별의 집', 'garden': '잊혀진 정원', 'plain': '그림자 평원', 'attic': '시간의 다락', 'hall': '별의 전당'}
def b64(p):
    buf = io.BytesIO(); Image.open(p).convert('RGB').resize((448, 252)).save(buf, 'JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()
cards = ''.join(
    f"<div style='background:#1e1b4b;border-radius:12px;padding:8px'><img src='data:image/jpeg;base64,{b64(out / 'web' / (x['name'] + '.webp'))}' style='width:100%;border-radius:8px'>"
    f"<div style='color:#fde68a;font-weight:700;margin-top:6px'>{LABELS[x['name']]}</div><div style='color:#c7d2fe;font-size:12px'>seed {x['seed']} · {x['seconds']}초</div></div>"
    for x in log['images'])
total = round(sum(x['seconds'] for x in log['images']), 2)
display(HTML(f"""
<div style='font-family:sans-serif;background:#0f0b2e;color:#eef2ff;padding:18px;border-radius:16px;max-width:1000px'>
  <div style='font-size:22px;font-weight:800;color:#fde68a'>✦ 별의 끝에서 — 교내 GPU(틸론) 이미지 생성 결과</div>
  <table style='margin:10px 0;font-size:14px;color:#eef2ff'>
    <tr><td style='padding-right:16px'>GPU</td><td><b>{log['gpu']}</b> ({log['gpu_total_gb']} GB)</td></tr>
    <tr><td>CUDA / torch</td><td>{log['cuda']} / {log['torch']}</td></tr>
    <tr><td>모델</td><td>{log['model']} · {W}×{H} · {STEPS} steps</td></tr>
    <tr><td>생성 시간</td><td>6장 합계 {total}초 · 최대 GPU 메모리 {log['peak_gpu_memory_gb']} GB</td></tr>
    <tr><td>실행 시각</td><td>{log['started_at']} ~ {log['finished_at']}</td></tr>
  </table>
  <div style='display:grid;grid-template-columns:repeat(3,1fr);gap:10px'>{cards}</div>
</div>"""))

# 같은 내용을 이미지 파일로도 저장 (발표 자료용, 글꼴 문제를 피하려고 영어 표기)
sheet = Image.new('RGB', (1400, 760), (15, 11, 46)); d = ImageDraw.Draw(sheet)
d.text((24, 18), f"At the End of Stars - generated on school GPU (Tilon)", fill=(253, 230, 138))
d.text((24, 42), f"GPU {log['gpu']} ({log['gpu_total_gb']}GB) | CUDA {log['cuda']} | {log['model']} {W}x{H} {STEPS} steps | total {total}s | peak {log['peak_gpu_memory_gb']}GB | {log['finished_at']}", fill=(199, 210, 254))
for i, x in enumerate(log['images']):
    im = Image.open(out / 'web' / (x['name'] + '.webp')).convert('RGB').resize((440, 248))
    cx, cy = 24 + (i % 3) * 456, 80 + (i // 3) * 336
    sheet.paste(im, (cx, cy)); d.text((cx, cy + 256), f"{x['name']}  seed {x['seed']}  {x['seconds']}s", fill=(253, 230, 138))
sheet.save(out / 'summary.png')
shutil.make_archive('starlight_outputs', 'zip', out)
print('요약 이미지 저장:', out / 'summary.png', '(zip에도 포함됨)')
